[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_10/Übungen/01_tag_10_uebung_prompt_experiment.ipynb)

# Übung: Kontrolliertes Prompt-Experiment für Bildgenerierung

Sie entwickeln für einen klaren Bildauftrag ein Prompt-Experiment. Statt viele zufällige Bilder zu erzeugen, verändern Sie jeweils **genau eine** Eigenschaft und bewerten anschließend systematisch, ob das Ergebnis den Auftrag erfüllt.

Sie können dafür entweder das lokale Stable-Diffusion-Modell aus Notebook 02 oder die OpenAI API aus Notebook 03 verwenden. Die Übung startet sicher ohne Bildgenerierung; API-Aufrufe müssen bewusst freigeschaltet werden.

## Lernziele

- einen Bildauftrag in überprüfbare Prompt-Bestandteile zerlegen,
- mit einem kontrollierten Experiment Ursache und Wirkung im Prompt untersuchen,
- lokale und API-basierte Bildgenerierung bewusst auswählen,
- Bildqualität anhand nachvollziehbarer Kriterien bewerten,
- Grenzen und Risiken eines generierten Ergebnisses reflektieren.

**Mindestumfang:** ein Basisprompt, zwei kontrollierte Varianten, drei erzeugte Bilder und eine ausgefüllte Bewertungstabelle. Bei Nutzung der API genügt auch ein Basisprompt plus eine Variante, um Kosten gering zu halten.

## Aufgabe 1 – Bildbriefing festlegen

Wählen Sie einen kleinen, unbedenklichen Bildauftrag. Gute Beispiele sind eine Buchillustration, ein Plakat ohne Marken, ein Reisebild, ein Fantasy-Gegenstand oder ein Tier in einer Szene. Vermeiden Sie reale Personen, geschützte Marken und Logos.

Formulieren Sie einen Auftrag mit fünf Bestandteilen: **Motiv**, **Umgebung**, **Komposition**, **Stil/Licht** und **Ausschlüsse**.

**Auszufüllen:**

- Zweck des Bildes: ____________________
- Motiv: ____________________
- Umgebung: ____________________
- Komposition: ____________________
- Stil und Licht: ____________________
- Ausschlüsse, etwa Text oder Wasserzeichen: ____________________

Antwort:

## Aufgabe 2 – Kontrollierte Varianten planen

Ein fairer Vergleich verändert nicht alles zugleich. Verwenden Sie einen Basisprompt und ändern Sie dann jeweils **nur eine** Spalte. Für die lokale Variante bleibt zusätzlich der Seed identisch, damit sich Änderungen eher auf den Prompt zurückführen lassen.

| Bild | Was bleibt gleich? | Einzige Änderung | Erwartete Wirkung |
| --- | --- | --- | --- |
| A: Basis | gesamtes Briefing | – | Referenzbild |
| B: Variante Licht | Motiv, Szene, Stil, Komposition | nur Licht | andere Stimmung, gleiches Motiv |
| C: Variante Komposition | Motiv, Szene, Stil, Licht | nur Kameraperspektive | anderes Bildlayout |

Schreiben Sie vor dem Generieren für Bild B und C Ihre Erwartung auf. Danach können Sie prüfen, ob das Modell die eine gewünschte Änderung tatsächlich isoliert umgesetzt hat.

In [ ]:
# Bearbeiten Sie zunächst nur die Textbausteine.
SUBJECT = 'a small red fox reading a book'
SETTING = 'in a cozy old library'
STYLE = 'detailed digital illustration'
NEGATIVE = 'no text, no watermark, no logo, no distorted anatomy'

base_prompt = f'{SUBJECT}, {SETTING}, eye-level composition, warm evening light, {STYLE}, {NEGATIVE}'
light_variant = f'{SUBJECT}, {SETTING}, eye-level composition, cool moonlight, {STYLE}, {NEGATIVE}'
composition_variant = f"{SUBJECT}, {SETTING}, bird's-eye view, warm evening light, {STYLE}, {NEGATIVE}"

PROMPTS = [
    ('A_basis', base_prompt),
    ('B_licht', light_variant),
    ('C_komposition', composition_variant),
]

for name, prompt in PROMPTS:
    print(f'{name}:\n  {prompt}\n')

## Aufgabe 3 – Erzeugungsweg wählen

| Weg | Wann passend? | Vor dem Start nötig |
| --- | --- | --- |
| `local` | Sie möchten mit Ihrer RTX 3080 arbeiten und das Stable-Diffusion-Modell aus Notebook 02 liegt schon lokal vor. | Modell in `saved_models/stable-diffusion-v1-5/` |
| `api` | Sie möchten das OpenAI-API-Beispiel aus Notebook 03 nutzen. | `.env` mit `API_TOKEN`; der Aufruf ist kostenpflichtig |

Die nächste Zelle macht zunächst nichts. Setzen Sie `RUN_GENERATION = True` erst, wenn Prompt und Methode überprüft sind. Für die API werden standardmäßig nur die ersten zwei Bilder erzeugt, damit keine unbeabsichtigten Kosten entstehen.

In [ ]:
# Sicherheits- und Arbeitskonfiguration
METHOD = 'local'  # 'local' oder 'api'
RUN_GENERATION = False  # Erst bewusst auf True setzen.
CONFIRM_API_COSTS = False  # Für METHOD = 'api' zusätzlich bewusst auf True setzen.
LOCAL_SEED = 42
LOCAL_STEPS = 30

# API: höchstens zwei Bilder als Mindestumfang; lokal standardmäßig alle drei.
active_prompts = PROMPTS if METHOD == 'local' else PROMPTS[:2]
print('Methode:', METHOD)
print('Bilder in diesem Durchlauf:', len(active_prompts))
print('Bildgenerierung aktiviert:', RUN_GENERATION)

## Aufgabe 4 – Bilder erzeugen

Führen Sie diese Zelle erst nach der Planung aus. Die lokale Route lädt Stable Diffusion aus dem lokalen Modellordner. Die API-Route liest den Schlüssel aus `.env`; der Schlüssel wird nicht angezeigt. Beide Routen speichern Ergebnisse in `notebooks/Day_10/generated_images/uebung/`.

In [ ]:
if not RUN_GENERATION:
    print('Noch kein Bild erzeugt. Prüfe Prompts und setze RUN_GENERATION = True, wenn du bereit bist.')
else:
    import base64
    import os
    from io import BytesIO
    from pathlib import Path

    import matplotlib.pyplot as plt
    import torch
    from PIL import Image

    COURSE_ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / 'requirements.txt').exists()), Path.cwd())
    DAY_DIR = COURSE_ROOT / 'notebooks' / 'Day_10'
    OUTPUT_DIR = DAY_DIR / 'generated_images' / 'uebung'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    generated = []

    if METHOD == 'local':
        from diffusers import StableDiffusionPipeline

        if not torch.cuda.is_available():
            raise RuntimeError("Die lokale Route benötigt CUDA. Wähle alternativ METHOD = 'api'.")
        model_dir = DAY_DIR / 'saved_models' / 'stable-diffusion-v1-5'
        if not model_dir.exists():
            raise FileNotFoundError("Lokales Modell fehlt. Führe zuerst Notebook 02 aus oder wähle METHOD = 'api'.")

        pipe = StableDiffusionPipeline.from_pretrained(
            str(model_dir), torch_dtype=torch.float16, local_files_only=True
        ).to('cuda')
        for name, prompt in active_prompts:
            generator = torch.Generator(device='cuda').manual_seed(LOCAL_SEED)
            image = pipe(
                prompt=prompt,
                negative_prompt=NEGATIVE,
                width=512, height=512,
                num_inference_steps=LOCAL_STEPS,
                guidance_scale=7.5,
                generator=generator,
            ).images[0]
            output_path = OUTPUT_DIR / f'{name}_local_seed_{LOCAL_SEED}.png'
            image.save(output_path)
            generated.append((name, image, output_path))

    elif METHOD == 'api':
        if not CONFIRM_API_COSTS:
            raise RuntimeError('Für die API bitte CONFIRM_API_COSTS = True setzen.')
        from dotenv import load_dotenv
        from openai import OpenAI

        env_path = COURSE_ROOT / '.env'
        load_dotenv(env_path)
        api_token = os.environ.get('API_TOKEN')
        if not api_token or api_token == 'xxxxx':
            raise ValueError('API_TOKEN fehlt. Prüfe die lokale .env-Datei.')
        client = OpenAI(api_key=api_token)
        for name, prompt in active_prompts:
            response = client.images.generate(
                model='gpt-image-2', prompt=prompt, size='1024x1024'
            )
            image = Image.open(BytesIO(base64.b64decode(response.data[0].b64_json)))
            output_path = OUTPUT_DIR / f'{name}_api.png'
            image.save(output_path)
            generated.append((name, image, output_path))

    else:
        raise ValueError("METHOD muss 'local' oder 'api' sein.")

    fig, axes = plt.subplots(1, len(generated), figsize=(5 * len(generated), 5))
    if len(generated) == 1:
        axes = [axes]
    for axis, (name, image, output_path) in zip(axes, generated):
        axis.imshow(image)
        axis.set_title(name)
        axis.axis('off')
        print(f'{name}: {output_path.resolve()}')
    plt.tight_layout()

## Aufgabe 5 – Qualität bewerten

Bewerten Sie jedes Bild mit `0` (nicht erfüllt), `1` (teilweise erfüllt) oder `2` (klar erfüllt). Ein Bild mit hoher technischer Qualität erfüllt den Briefing-Auftrag nicht automatisch.

| Bild | Motiv korrekt | gewünschte Änderung isoliert | Komposition | keine störenden Artefakte | kein Text/Logo | Summe / 10 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| A: Basis |  | – |  |  |  |  |
| B: Licht |  |  |  |  |  |  |
| C: Komposition |  |  |  |  |  |  |

**Auswertung:**

1. Welche Änderung hat das Modell am zuverlässigsten umgesetzt?
2. Wo hat sich mehr verändert als beabsichtigt?
3. Wie würdest du den Prompt für einen weiteren Durchlauf gezielt verbessern?
4. Weshalb hast du die lokale oder API-basierte Variante gewählt?

Antwort:

## Zusatzaufgabe – Vergleich der Wege

Erzeuge denselben Basisprompt einmal lokal und einmal über die API – nur wenn du für die API-Ausgaben Budget eingeplant hast. Vergleiche Prompttreue, Bildstil, Geschwindigkeit, Datenschutz, Kosten und Reproduzierbarkeit.

| Kriterium | Lokal: Stable Diffusion | API: OpenAI |
| --- | --- | --- |
| Prompttreue |  |  |
| Bildqualität |  |  |
| Ausführungszeit |  |  |
| Kosten |  |  |
| Datenschutz / Datenweg |  |  |
| Reproduzierbarkeit |  |  |

Für welches Szenario würdest du welchen Weg empfehlen – und warum?